# Chapter 7 (심화) — DPO 로 선호도 학습하기 : 코드 해설 노트북 (Python 초보용)

> `Chapter_7_Exercise_Follow_Instructions_dpo.ipynb` 의 원본 셀은 그대로 두고, 번호가 붙은 셀마다 바로 위에 설명을 넣은 해설판입니다. 셀 1이 아주 길어서 설명을 **함수 단위**로 나누었습니다.

## 이 노트북에서 하는 일 (한 문장)

같은 질문에 대한 **"좋은 답"과 "나쁜 답"을 짝지어 보여 주고**, 모델이 좋은 답 쪽을 더 선호하도록 가르칩니다. 이 방법의 이름이 **DPO(Direct Preference Optimization, 직접 선호도 최적화)** 입니다.

## 앞 노트북(SFT)과 무엇이 다른가

| | SFT (앞 노트북) | DPO (이 노트북) |
|---|---|---|
| 보여 주는 것 | 모범 답 **1개** | 좋은 답 + 나쁜 답 **한 쌍** |
| 가르치는 것 | "이렇게 답해라" | "**이것보다 저것이 낫다**" |
| 필요한 모델 | 1개 | **2개** (학습할 모델 + 기준 모델) |
| 비유 | 모범 답안을 따라 쓰기 | 두 답안을 비교하며 "이쪽이 더 좋아"라고 첨삭받기 |

말투·공손함·안전성처럼 **"정답이 하나로 정해지지 않고 더 나은 쪽이 있을 뿐"** 인 성질을 가르칠 때 쓰는 방법입니다.

## 데이터는 이렇게 생겼습니다

`data/instruction-data-with-preference.json` (1,100개). 앞 노트북 데이터에 `chosen`, `rejected` 두 칸이 추가되었습니다.

```json
{
  "instruction": "Edit the following sentence for grammar.",
  "input": "He go to the park every day.",
  "output": "He goes to the park every day.",
  "chosen": "He goes to the park every day.",
  "rejected": "He goes to the stupid park every single day."
}
```
| 키 | 뜻 |
|---|---|
| `chosen` | 👍 선호하는(골라야 할) 답 — 공손한 말투 |
| `rejected` | 👎 선호하지 않는(피해야 할) 답 — 내용은 맞지만 무례한 말투 |

두 답 모두 문법적으로는 "정답"입니다. 차이는 **말투**뿐이며, 이 노트북의 목표는 모델이 공손한 쪽을 고르게 만드는 것입니다.

## 두 모델의 역할

| 이름 | 역할 | 학습 여부 |
|---|---|---|
| **policy model** (정책 모델) | 우리가 고쳐 나가는 모델 | 학습됨 |
| **reference model** (기준 모델) | "원래는 어땠는지"를 알려 주는 비교 기준 | **고정** (변하지 않음) |

둘은 **똑같은 가중치**로 출발합니다. 기준 모델이 필요한 이유는, 학습 중인 모델이 원래 모습에서 **너무 멀리 벗어나 말하는 능력 자체가 망가지는 것**을 막기 위해서입니다.

## DPO 의 핵심 아이디어

```
 policy    :  logP(chosen) - logP(rejected)   = how much policy    prefers chosen
 reference :  logP(chosen) - logP(rejected)   = how much reference prefers chosen

 goal :  make the policy's preference LARGER than the reference's
```

## 셀 구성

| 셀 | 내용 |
|---|---|
| 1 | 함수·클래스 정의 : 데이터 준비, `compute_logprobs` ⭐, `compute_dpo_loss` ⭐, 학습 루프 |
| 2 | 실행 : 데이터 → 모델 2개 → 학습 |
| 3 | 손실 그래프 |
| 4 | 기준 모델과 학습된 모델의 답변 비교 |
| 5 | (빈 셀) |

### ⚠️ 실행 전 꼭 알아둘 점 — 어떤 가중치를 불러오는가
셀 2의 주석에는 "SFT 모델 로드"라고 적혀 있지만, 실제 코드가 불러오는 파일은 `gpt2-medium-355M.pth`, 즉 **지시 미세 조정을 하지 않은 원본 GPT-2 Medium** 입니다. DPO 는 원래 **앞 노트북에서 SFT 를 마친 모델**(지시에 답할 줄 아는 모델)을 출발점으로 삼는 방법입니다. 원본 GPT-2 는 지시문을 이해하지 못하므로, 셀 4에 저장된 출력에서 두 모델 모두 엉뚱한 글을 반복하는 것이 이 때문입니다. 자세한 내용은 셀 2, 셀 4 설명에 적었습니다.

---
## 셀 1 — 함수와 클래스 정의

### 🎯 이 셀이 하는 일
DPO 학습에 필요한 모든 도구를 **정의**합니다. 실행해도 출력은 없습니다. 코드 속 `# 1.` ~ `# 4.` 구역 주석을 따라 읽으면 됩니다.

| 구역 | 이름 | 한 줄 역할 |
|---|---|---|
| 1 | `download_and_load_file` | JSON 데이터 내려받아 읽기 |
| 1 | `format_input` | 프롬프트 양식 만들기 (앞 노트북과 동일) |
| 2 | `PreferenceDataset` | (프롬프트, 좋은 답, 나쁜 답)을 토큰화해 보관 |
| 2 | `custom_collate_fn` | 길이를 맞추고 **채점할 자리 표시(마스크)** 만들기 |
| 3 | `compute_logprobs` | 모델이 어떤 답에 준 **평균 로그 확률** 계산 ⭐ |
| 3 | `compute_dpo_loss` | DPO 손실 공식 ⭐ |
| 3 | `compute_dpo_loss_batch` | 배치 하나에 대해 위 둘을 묶어 실행 |
| 4 | `evaluate_dpo_loss_loader` | 중간 점검 |
| 4 | `train_model_dpo_simple` | 학습 루프 |

### import 중 새로운 것
```python
import torch.nn.functional as F
```
`torch.nn.functional` 을 `F` 라는 짧은 별명으로 부릅니다. `F.log_softmax`, `F.logsigmoid` 처럼 씁니다.

```python
try:
    from previous_chapters import GPTModel, generate_and_print_sample, download_model
except ImportError:
    print("경고: 'previous_chapters' 모듈을 찾을 수 없습니다. ...")
```
`try / except ImportError` : 모듈을 불러오다 실패해도 프로그램을 멈추지 않고 경고만 출력합니다. (단, 실패하면 뒤에서 `GPTModel` 을 쓸 때 결국 에러가 납니다.)

---
### ① `download_and_load_file`

```python
if not os.path.exists(file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(response.text)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
return json.loads(text_data)
```
- `os.path.dirname("data/파일.json")` → `"data"` (경로에서 폴더 부분만). 그 폴더가 없으면 만듭니다.
- `json.loads(문자열)` : JSON **글자**를 파이썬 리스트/딕셔너리로 바꿉니다. (`json.load` 는 파일에서, `json.loads` 는 문자열에서 읽습니다. 끝의 s 는 string)

> 📝 **코드의 빈틈** : 파일이 **없어서 새로 내려받는 경우**(`if` 쪽)에는 `text_data` 라는 변수를 만들지 않습니다. 그래서 마지막 줄 `json.loads(text_data)` 에서 "그런 변수가 없다"는 에러가 납니다. 지금은 `data/` 폴더에 파일이 이미 있어서 `else` 쪽으로 가므로 문제없이 동작합니다. 만약 에러를 만나면 **셀을 한 번 더 실행**하면 됩니다(두 번째에는 파일이 있으므로). 고치려면 `if` 블록 안에 `text_data = response.text` 한 줄을 넣으면 됩니다.

### ② `format_input`
앞 노트북과 **완전히 같습니다.** 항목을 `### Instruction:` / `### Input:` 표지판이 붙은 Alpaca 양식 프롬프트로 바꿉니다.

---
### ③ `PreferenceDataset`

```python
for entry in data:
    prompt = format_input(entry)
    rejected_response = entry["rejected"]
    chosen_response = entry["chosen"]

    prompt_tokens = tokenizer.encode(prompt)
    chosen_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{chosen_response}")
    rejected_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{rejected_response}")

    self.encoded_texts.append({
        "prompt": prompt_tokens,
        "chosen": chosen_full_tokens,
        "rejected": rejected_full_tokens,
    })
```
항목 하나마다 **세 가지**를 토큰화해 딕셔너리로 묶어 보관합니다.

```
 "prompt"   : [ prompt ]
 "chosen"   : [ prompt ][ ### Response: ][ good answer ]
 "rejected" : [ prompt ][ ### Response: ][ bad answer  ]
```
프롬프트만 따로 보관하는 이유는 **프롬프트가 몇 토큰인지** 알아야 나중에 그 부분을 채점에서 뺄 수 있기 때문입니다.

`__getitem__` 은 이 딕셔너리 하나를, `__len__` 은 항목 수를 돌려줍니다.

---
### ④ `custom_collate_fn` — 배치 묶기 + 마스크 만들기

앞 노트북의 collate 함수는 "채점하지 않을 자리"를 정답에 −100 을 써넣어 표시했습니다. 여기서는 **마스크(True/False 표)** 를 따로 만들어 표시합니다.

> **마스크** : 토큰마다 "이 자리를 채점에 **포함할까(True)** / **뺄까(False)**"를 적어 둔 표

**1. 배치에서 가장 긴 길이 찾기**
```python
max_length_common = 0
if batch:
    for key in ["chosen", "rejected"]:
        current_max = max(len(item[key]) + 1 for item in batch)
        max_length_common = max(max_length_common, current_max)
```
- `if batch:` : 배치가 비어 있지 않을 때만 (빈 리스트는 거짓)
- 좋은 답과 나쁜 답을 **통틀어** 가장 긴 것 + 1 을 공통 길이로 정합니다. 두 종류의 길이를 똑같이 맞추기 위해서입니다.

**2. 패딩과 마스크**
```python
for item in batch:
    prompt = torch.tensor(item["prompt"])
    batch_data["prompt"].append(prompt)

    for key in ["chosen", "rejected"]:
        sequence = item[key]
        padded = sequence + [pad_token_id] * (max_length_common - len(sequence))

        mask = torch.ones(len(padded)).bool()
        mask[len(sequence):] = False

        if mask_prompt_tokens:
            mask[:prompt.shape[0] + 2] = False

        batch_data[key].append(torch.tensor(padded))
        batch_data[f"{key}_mask"].append(mask)
```

| 코드 | 설명 |
|---|---|
| `padded = sequence + [50256] * (...)` | 모자란 만큼 뒤에 패딩 토큰 채우기 |
| `torch.ones(n).bool()` | 전부 True 인 마스크로 시작 |
| `mask[len(sequence):] = False` | **패딩 자리**는 False |
| `mask[:prompt.shape[0] + 2] = False` | **프롬프트 자리**(와 그 바로 뒤 구분 부분 2토큰)도 False |
| `f"{key}_mask"` | `"chosen_mask"` 또는 `"rejected_mask"` 라는 이름을 만들어 냄 |

```
 tokens : [ prompt ............ ][ response ........ ][ pad pad ]
 mask   :   F  F  F  F  F  F  F    T  T  T  T  T  T     F   F
```

**⭐ 왜 프롬프트를 가릴까?** 좋은 답과 나쁜 답은 **프롬프트 부분이 완전히 같습니다.** 우리가 비교하려는 것은 "답변 부분을 얼마나 그럴듯하게 여기는가"이므로, 질문 부분은 채점에서 빼고 **답변 부분만** 봅니다.

**3. 텐서로 묶기**
```python
for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
    tensor_stack = torch.stack(batch_data[key])
    if allowed_max_length is not None:
        tensor_stack = tensor_stack[:, :allowed_max_length]
    batch_data[key] = tensor_stack.to(device)
return batch_data
```
네 가지를 각각 `[배치 크기, 길이]` 텐서로 쌓고, 너무 길면 자르고, GPU 로 보냅니다. 돌려주는 것은 텐서 하나가 아니라 **딕셔너리**입니다 → 나중에 `batch["chosen"]`, `batch["chosen_mask"]` 처럼 꺼내 씁니다.

---
### ⑤ `compute_logprobs` — "모델이 이 답을 얼마나 그럴듯하게 보는가" (⭐)

**하는 일** : 어떤 문장(좋은 답 또는 나쁜 답)을 모델에 넣었을 때, 모델이 **그 문장의 실제 토큰들에 준 로그 확률의 평균**을 구합니다. 값이 클수록(0 에 가까울수록) 모델이 그 답을 자연스럽게 여긴다는 뜻입니다.

**로그 확률이란?** 확률에 로그를 씌운 값입니다.
```
 prob 1.0   ->  log  0.0
 prob 0.5   ->  log -0.69
 prob 0.01  ->  log -4.6        always <= 0, closer to 0 = more likely
```
확률을 여러 번 곱하면 숫자가 너무 작아지지만, 로그를 쓰면 **더하기**로 바뀌어 다루기 쉽습니다.

**1. 한 칸 밀어 짝 맞추기**
```python
labels = labels[:, 1:].clone()
logits = logits[:, :-1, :]
```
모델의 i 번째 자리 출력은 **i+1 번째 토큰**에 대한 예측입니다. 그래서 정답은 앞 하나를 빼고(`1:`), 예측은 뒤 하나를 빼서(`:-1`) 짝을 맞춥니다. `.clone()` 은 원본을 건드리지 않게 복사본을 만드는 것입니다.

```
 input tokens :   A    B    C    D
 logits       :  [B?] [C?] [D?] [E?]   -> drop last   : [B?] [C?] [D?]
 labels       :   A    B    C    D     -> drop first  :  B    C    D
```

**2. 점수를 로그 확률로**
```python
log_probs = F.log_softmax(logits, dim=-1)
```
softmax(확률로 변환) 후 로그를 한 번에 계산합니다. 모양 `[b, T-1, 50257]`.

**3. 실제 토큰의 값만 뽑기 — `gather`**
```python
selected_log_probs = torch.gather(
    input=log_probs, dim=-1, index=labels.unsqueeze(-1)
).squeeze(-1)
```
자리마다 5만여 개의 로그 확률이 있는데, 그중 **실제로 그 자리에 온 토큰 하나의 값**만 필요합니다. `gather` 는 "각 줄에서 `index` 가 가리키는 칸의 값만 꺼내라"는 함수입니다.

```
 log_probs (3 positions x 4 vocab)        labels      gathered
 pos0  [-1.2, -0.3, -2.5, -3.0]             1    ->    -0.3
 pos1  [-0.1, -4.0, -2.2, -3.1]             0    ->    -0.1
 pos2  [-2.0, -1.5, -0.4, -2.8]             2    ->    -0.4
```
- `labels.unsqueeze(-1)` : `gather` 가 요구하는 모양에 맞추려고 맨 뒤에 축을 하나 추가
- `.squeeze(-1)` : 꺼낸 뒤 그 축을 다시 제거 → 모양 `[b, T-1]`
- 빈칸 문제의 요점 : 원래 점수(`logits`)가 아니라 **`log_probs`** 에서 꺼내야 합니다.

**4. 마스크 적용 후 평균**
```python
if selection_mask is not None:
    mask = selection_mask[:, 1:].clone()
    selected_log_probs = selected_log_probs * mask
    avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
    return avg_log_prob
else:
    return selected_log_probs.mean(-1)
```
- 마스크도 정답과 똑같이 한 칸 밉니다.
- `* mask` : True 는 1, False 는 0 으로 계산되므로 **가린 자리의 값이 0** 이 됩니다.
- `.sum(-1) / mask.sum(-1)` : 남은 값들의 합 ÷ **채점한 자리 수** = 답변 부분만의 평균
- 결과 모양 `[b]` — 샘플 하나당 숫자 하나

---
### ⑥ `compute_dpo_loss` — DPO 손실 공식 (⭐ 이 노트북의 핵심)

입력은 숫자 네 묶음입니다.

| | 좋은 답(chosen) | 나쁜 답(rejected) |
|---|---|---|
| 학습 중인 모델 | `model_chosen_logprobs` | `model_rejected_logprobs` |
| 기준 모델 | `reference_chosen_logprobs` | `reference_rejected_logprobs` |

```python
model_logratios = model_chosen_logprobs - model_rejected_logprobs
reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
logits = model_logratios - reference_logratios
losses = -F.logsigmoid(beta * logits)
```

| 줄 | 의미 |
|---|---|
| `model_logratios` | 학습 중인 모델이 좋은 답을 나쁜 답보다 **얼마나 더 선호하는가** |
| `reference_logratios` | 기준 모델은 얼마나 더 선호**했는가** |
| `logits` (둘의 차이) | 기준보다 선호가 **얼마나 더 커졌는가** — 이 값을 키우는 것이 목표 |
| `-F.logsigmoid(beta * logits)` | 그 값이 클수록 **손실이 작아지도록** 바꾼 것 |

**숫자로 보기** (`beta = 0.1`)
```
 logits = -20  ->  loss 2.13     worse than reference
 logits =   0  ->  loss 0.693    same as reference   <- starting point
 logits = +20  ->  loss 0.13     prefers chosen much more
```
- **학습 시작 시 손실은 약 0.693** 입니다. 두 모델이 똑같이 출발하므로 차이가 0 이고, −log(0.5) = 0.693 이기 때문입니다. 이 숫자에서 **내려가면** 학습이 되고 있는 것입니다.
- `sigmoid` 는 어떤 숫자든 0~1 사이로 눌러 주는 함수이고, `logsigmoid` 는 거기에 로그를 씌운 것입니다.
- **`beta`** : 차이를 얼마나 민감하게 반영할지 정하는 값입니다. 클수록 기준 모델과의 차이에 예민해집니다.

**왜 기준 모델의 값을 빼 줄까?** 그냥 "좋은 답 확률을 높이고 나쁜 답 확률을 낮춰라"만 시키면, 모델이 말하는 능력을 망가뜨리면서까지 숫자만 벌릴 수 있습니다. "**원래 모델에 비해** 얼마나 나아졌는가"를 기준으로 삼아 원래 모습을 붙잡아 두는 역할을 합니다.

```python
chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()
return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()
```
- `chosen_rewards` : 좋은 답의 확률이 기준보다 얼마나 **올랐는지** (양수면 좋음)
- `rejected_rewards` : 나쁜 답의 확률이 기준보다 얼마나 올랐는지 (**음수**면 좋음)
- `.detach()` : 학습에는 쓰지 않고 **구경(로그)용**으로만 쓰겠다는 표시
- `.mean()` : 배치 평균. 세 값을 함께 돌려줍니다.

---
### ⑦ `compute_dpo_loss_batch`

```python
policy_chosen_log_probas = compute_logprobs(
    logits=policy_model(batch["chosen"]),
    labels=batch["chosen"],
    selection_mask=batch["chosen_mask"]
)
policy_rejected_log_probas = compute_logprobs(logits=policy_model(batch["rejected"]), ...)

with torch.no_grad():
    ref_chosen_log_probas = compute_logprobs(logits=reference_model(batch["chosen"]), ...)
    ref_rejected_log_probas = compute_logprobs(logits=reference_model(batch["rejected"]), ...)

loss, chosen_rewards, rejected_rewards = compute_dpo_loss(...)
```
배치 하나에 대해 **모델을 네 번** 실행합니다.

```
                  chosen      rejected
 policy model     run #1      run #2       <- with gradients (trainable)
 reference model  run #3      run #4       <- torch.no_grad() (fixed)
```
- `labels=batch["chosen"]` : 입력과 정답이 **같은 문장**입니다. `compute_logprobs` 안에서 한 칸 밀어 쓰기 때문입니다.
- 기준 모델은 학습하지 않으므로 `torch.no_grad()` 로 감싸 메모리를 아낍니다.
- 일반 학습보다 **계산이 약 4배** 들고 모델도 2개를 메모리에 올려야 해서 꽤 무겁습니다.

---
### ⑧ `evaluate_dpo_loss_loader` — 중간 점검

```python
policy_model.eval()
with torch.no_grad():
    def compute_loader_metric(loader):
        total_loss, total_chosen, total_rejected = 0., 0., 0.
        num_batches = min(eval_iter, len(loader))
        if num_batches == 0: return float("nan"), float("nan"), float("nan")
        for i, batch in enumerate(loader):
            if i >= num_batches: break
            loss, chosen, rejected = compute_dpo_loss_batch(batch, policy_model, reference_model, beta)
            total_loss += loss.item()
            ...
        return total_loss/num_batches, total_chosen/num_batches, total_rejected/num_batches

    train_loss, train_chosen, train_rejected = compute_loader_metric(train_loader)
    val_loss, val_chosen, val_rejected = compute_loader_metric(val_loader)
policy_model.train()
return { "train_loss": train_loss, ... }
```
- **함수 안에 함수**(`compute_loader_metric`)를 정의했습니다. 학습용·검증용 로더에 **같은 계산을 두 번** 해야 해서, 그 부분을 작은 함수로 묶어 재사용한 것입니다.
- `if 조건: return ...` / `if 조건: break` : 실행할 문장이 한 줄이면 이렇게 **같은 줄에** 쓸 수 있습니다.
- 앞에서 배운 `eval()` → `no_grad()` → `train()` 샌드위치 구조 그대로입니다.
- 결과 6개를 **딕셔너리**로 돌려줍니다. 값이 많을 때는 이름표를 붙여 돌려주는 편이 헷갈리지 않습니다.

---
### ⑨ `train_model_dpo_simple` — 학습 루프

```python
tracking = {"train_losses": [], "val_losses": [], "tokens_seen": []}
tokens_seen, global_step = 0, -1

for epoch in range(num_epochs):
    policy_model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
            batch=batch, policy_model=policy_model, reference_model=reference_model, beta=beta)
        loss.backward()
        optimizer.step()
        tokens_seen += batch["chosen"].numel()
        global_step += 1
        if global_step % eval_freq == 0:
            res = evaluate_dpo_loss_loader(...)
            ...
            print(f"Ep {epoch+1} (Step {global_step:06d}): "
                  f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                  f"Train Margin {res['train_chosen_reward']-res['train_rejected_reward']:.3f}")
    generate_and_print_sample(model=policy_model, tokenizer=tokenizer,
                              device=loss.device, start_context=start_context)
return tracking
```
뼈대는 Chapter 5 와 **똑같습니다** (`zero_grad → loss → backward → step`). 달라진 것은 손실을 구하는 함수가 `compute_dpo_loss_batch` 로 바뀐 것뿐입니다.

- 기록장을 리스트 3개 대신 **딕셔너리 하나**(`tracking`)에 담았습니다. `tracking["train_losses"].append(...)` 처럼 씁니다.
- 옵티마이저에는 `policy_model` 의 가중치만 들어 있으므로(셀 2), 기준 모델은 절대 바뀌지 않습니다.
- **Train Margin** = 좋은 답 보상 − 나쁜 답 보상. **클수록** 모델이 좋은 답과 나쁜 답을 잘 구분하게 됐다는 뜻입니다.
- f-문자열 안에서 딕셔너리 값을 꺼낼 때는 바깥 따옴표(`"`)와 **다른 따옴표**(`'`)를 씁니다 : `{res['train_loss']:.3f}`
- `device=loss.device` : 손실 텐서가 있는 장치를 그대로 알려 주는 요령입니다.

In [1]:
# 1
import os
import json
import time
import requests
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from functools import partial
import tiktoken

# ==========================================
# [중요] 외부 모듈 의존성 처리
# 이 코드는 책이나 튜토리얼의 일부로, 이전 챕터에서 정의된 모델(GPTModel)이 필요합니다.
# ==========================================
try:
    from previous_chapters import GPTModel, generate_and_print_sample, download_model
except ImportError:
    print("경고: 'previous_chapters' 모듈을 찾을 수 없습니다. GPTModel 등의 정의가 필요합니다.")

# ==========================================
# 1. 설정 및 데이터 로드 유틸리티
# 데이터를 인터넷에서 다운로드하거나 로컬에서 불러오는 기능을 담당합니다.
# ==========================================
def download_and_load_file(file_path, url):
    """
    지정된 경로에 파일이 없으면 URL에서 다운로드하고,
    파일을 읽어 JSON 데이터를 반환합니다.
    """
    if not os.path.exists(file_path):
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(response.text)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()
    return json.loads(text_data)


def format_input(entry):
    """
    JSON 데이터의 항목을 모델이 이해할 수 있는 프롬프트 형식으로 변환합니다.
    (알파카(Alpaca) 데이터셋 스타일 포맷팅)
    """
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    # 추가 입력(Input)이 있는 경우에만 붙임
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

# ==========================================
# 2. 데이터셋 및 Collate 함수 (DPO용)
# DPO는 (프롬프트, 선택된 답변, 거부된 답변)의 쌍이 필요합니다.
# ==========================================
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        
        # 데이터를 미리 토큰화(Tokenization)하여 저장합니다.
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"] # 👎 덜 선호되는 답변
            chosen_response = entry["chosen"]     # 👍 더 선호되는 답변

            # 프롬프트 부분만 따로 인코딩 (나중에 마스킹하기 위해 필요)
            prompt_tokens = tokenizer.encode(prompt)
            
            # 프롬프트 + 답변 형태로 전체 문장을 인코딩
            # TODO: 선호/비선호 응답 변수명을 채우세요.
            # 힌트: 바로 위에서 만든 chosen_response와 rejected_response를 Response 뒤에 붙입니다.
            # 예시: chosen 쪽은 chosen_response, rejected 쪽은 rejected_response를 사용합니다.
            # chosen_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")
            chosen_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{chosen_response}")
            # rejected_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")
            rejected_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{rejected_response}")

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

def custom_collate_fn(batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu"):
    """
    배치(Batch) 내의 데이터 길이를 맞추고(Padding), 마스크(Mask)를 생성하는 함수입니다.
    """
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [], # Loss 계산 시 무시할 부분 (패딩 등)
        "chosen_mask": []
    }

    # 1. 배치 내에서 가장 긴 시퀀스 길이 찾기 (패딩을 위해)
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key]) + 1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # 2. 패딩 추가 및 마스크 생성
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            sequence = item[key]
            # 최대 길이에 맞춰 패딩 토큰 추가
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            
            # 기본 마스크: 데이터가 있는 곳은 1(True), 패딩은 0(False)
            mask = torch.ones(len(padded)).bool()
            mask[len(sequence):] = False  # 패딩 부분 마스킹

            # [중요] 프롬프트 부분 마스킹 (선택적)
            # DPO는 '답변'의 확률 차이를 학습하므로, 질문(Prompt) 부분은 Loss 계산에서 제외합니다.
            # +2는 \n\n때문
            if mask_prompt_tokens:
                mask[:prompt.shape[0] + 2] = False 

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 3. 텐서 변환 및 디바이스(GPU/CPU) 이동
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = torch.stack(batch_data[key])
        # 길이 제한이 있다면 자르기
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        batch_data[key] = tensor_stack.to(device)

    return batch_data

# ==========================================
# 3. DPO Loss 및 Log Probability 계산 함수
# DPO의 핵심 수학적 로직이 들어있는 부분입니다.
# ==========================================
def compute_logprobs(logits, labels, selection_mask=None):
    """
    모델의 출력(logits)과 정답(labels)을 받아 해당 정답 토큰의 로그 확률을 계산합니다.
    """
    # Auto-regressive 모델 특성상, 입력 [A, B, C]에 대해 예측은 [B, C, D]가 되므로 시프트(Shift)합니다.
    # TODO: 자동회귀 시프트 인덱스를 채우세요.
    # 힌트: labels는 1칸 앞당기고 logits는 마지막 시점을 제외해 정렬합니다.
    # labels = labels[:, ????:].clone()
    labels = labels[:, 1:].clone()
    # logits = logits[:, :????, :]
    logits = logits[:, :-1, :]
    
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 실제 정답 레이블에 해당하는 확률값만 추출 (gather 사용)
    # TODO: gather의 input 텐서를 채우세요.
    # 힌트: raw logits가 아닌 log_probs에서 정답 위치를 gather해야 합니다.
    selected_log_probs = torch.gather(
        # input=????,
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        # 마스크도 시프트하여 적용 (패딩이나 프롬프트 영역 무시)
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        
        # 유효한 토큰들의 로그 확률 평균 계산
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)

def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs, 
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    """
    DPO 손실 함수 계산:
    Policy 모델이 Reference 모델보다 'chosen' 답변을 더 선호하고, 'rejected' 답변을 덜 선호하도록 유도합니다.
    
    beta: Reference 모델에서 얼마나 벗어날지 제어하는 하이퍼파라미터 (보통 0.1~0.5)
    """
    # 모델의 (Chosen - Rejected) 로그 확률 차이
    # TODO: 정책 모델 chosen 로그확률 변수를 채우세요.
    # 힌트: chosen - rejected 순서로 빼야 preference 방향이 맞습니다.
    # model_logratios = ???? - model_rejected_logprobs
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    # 기준 모델의 (Chosen - Rejected) 로그 확률 차이
    # TODO: 레퍼런스 모델 chosen 로그확률 변수를 채우세요.
    # 힌트: reference도 chosen - rejected 순서를 동일하게 유지합니다.
    # reference_logratios = ???? - reference_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    
    # 두 비율의 차이 (Policy가 Reference보다 얼마나 더 잘 구분했는가)
    # TODO: DPO logits 차감 대상을 채우세요.
    # 힌트: policy 비율에서 reference 비율을 빼는 형태입니다.
    # logits = model_logratios - ????
    logits = model_logratios - reference_logratios
    
    # Sigmoid 후 음수 로그 (Cross Entropy와 유사) -> 이 값을 최소화하면 선호도 차이가 극대화됨
    # TODO: DPO 손실 함수명을 채우세요.
    # 힌트: beta * logits에 logsigmoid를 적용해 손실을 계산합니다.
    # losses = -F.????(beta * logits)
    losses = -F.logsigmoid(beta * logits)
    
    # 학습 추적용 보상(Reward) 계산 (실제 학습엔 안 쓰이고 로깅용)
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()
    
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """
    하나의 배치에 대해 전체 DPO 과정을 수행하는 헬퍼 함수
    """
    # 1. 학습 중인 모델(Policy Model)의 로그 확률 계산 (기울기 계산 O)
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )
    
    # 2. 기준 모델(Reference Model)의 로그 확률 계산 (기울기 계산 X -> 메모리 절약)
    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )

    # 3. 최종 Loss 계산
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

# ==========================================
# 4. 평가 및 학습 루프
# 실제 모델 학습을 돌리는 메인 루프입니다.
# ==========================================
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):
    """
    학습 중간에 모델 성능을 평가(Validation)하는 함수
    """
    policy_model.eval() # 평가 모드 전환 (Dropout 등 비활성화)
    with torch.no_grad():
        # 로더를 순회하며 평균 Loss 계산하는 내부 함수
        def compute_loader_metric(loader):
            total_loss, total_chosen, total_rejected = 0., 0., 0.
            num_batches = min(eval_iter, len(loader))
            if num_batches == 0: return float("nan"), float("nan"), float("nan")
            
            for i, batch in enumerate(loader):
                if i >= num_batches: break
                loss, chosen, rejected = compute_dpo_loss_batch(batch, policy_model, reference_model, beta)
                total_loss += loss.item()
                total_chosen += chosen.item()
                total_rejected += rejected.item()
            return total_loss/num_batches, total_chosen/num_batches, total_rejected/num_batches

        train_loss, train_chosen, train_rejected = compute_loader_metric(train_loader)
        val_loss, val_chosen, val_rejected = compute_loader_metric(val_loader)

    policy_model.train() # 다시 학습 모드로 전환
    return {
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen,
        "train_rejected_reward": train_rejected,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen,
        "val_rejected_reward": val_rejected
    }

def train_model_dpo_simple(policy_model, reference_model, train_loader, val_loader, 
                           optimizer, num_epochs, beta, eval_freq, eval_iter, start_context, tokenizer):
    """
    전체 학습 과정을 관리하는 함수
    """
    tracking = {"train_losses": [], "val_losses": [], "tokens_seen": []}
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        policy_model.train() # 학습 모드 시작
        for batch in train_loader:
            optimizer.zero_grad() # 이전 기울기 초기화
            
            # Loss 계산
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch, policy_model=policy_model, reference_model=reference_model, beta=beta
            )
            
            loss.backward() # 역전파 (기울기 계산)
            optimizer.step() # 가중치 업데이트
            
            tokens_seen += batch["chosen"].numel()
            global_step += 1

            # 평가 주기마다 성능 측정 및 출력
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model, reference_model, train_loader, val_loader, beta, eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["tokens_seen"].append(tokens_seen)
                
                # Margin = Chosen Reward - Rejected Reward (클수록 좋음, 모델이 정답을 오답보다 더 선호한다는 뜻)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                      f"Train Margin {res['train_chosen_reward']-res['train_rejected_reward']:.3f}")

        # 에포크가 끝날 때마다 샘플 생성하여 육안으로 확인
        generate_and_print_sample(model=policy_model, tokenizer=tokenizer, device=loss.device, start_context=start_context)

    return tracking

### 💡 주요 핵심 포인트

1. **DPO의 원리**: 일반적인 학습과 달리 **두 개의 모델**이 필요합니다.
* `policy_model`: 학습이 되어 업데이트되는 모델 (우리가 훈련시키는 모델).
* `reference_model`: 학습되지 않고 고정된 상태로 비교 기준이 되는 모델 (보통 SFT가 완료된 모델 사용).


2. **데이터 구성**: `(Prompt, Chosen, Rejected)` 세 가지가 한 세트입니다. 모델이 `Chosen`(선호) 쪽으로 확률을 높이고 `Rejected`(비선호) 쪽으로 확률을 낮추도록 유도합니다.
3. **마스킹(Masking)**: `custom_collate_fn`에서 프롬프트 부분을 `False`로 마스킹합니다. 모델이 질문(Prompt)을 생성하는 능력은 평가하지 않고, 질문에 대한 **답변(Response)** 부분의 확률만을 가지고 Loss를 계산하기 위함입니다.


---
## 셀 2 — 실행 : 데이터 → 모델 2개 → 학습

### 🎯 이 셀이 하는 일
셀 1의 도구들을 사용해 실제로 DPO 학습을 실행합니다.

### 코드 뜯어보기

**① 장치, 토크나이저, 데이터**
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
data = download_and_load_file(file_path, url)

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]
```
앞 노트북과 같은 비율(학습 935 / 테스트 110 / 검증 55)로 나눕니다.

**② 데이터 로더**
```python
customized_collate_fn = partial(
    custom_collate_fn, device=device, mask_prompt_tokens=True, allowed_max_length=1024
)
train_loader = DataLoader(PreferenceDataset(train_data, tokenizer), batch_size=batch_size,
                          collate_fn=customized_collate_fn, shuffle=True, drop_last=True)
```
- `partial` : `custom_collate_fn` 의 일부 인자를 미리 채운 새 함수를 만듭니다(앞 노트북에서 배운 것).
- `PreferenceDataset(...)` 을 변수에 담지 않고 `DataLoader` 의 인자 자리에서 **바로 만들어** 넘겼습니다.

**③ 모델 2개 준비**
```python
BASE_CONFIG = { ..., "emb_dim": 1024, "n_layers": 24, "n_heads": 16 }   # GPT-2 Medium

file_name = "gpt2-medium-355M.pth"
model_path = f"./models/gpt2/{file_name}"
download_model(file_name, model_path)

policy_model = GPTModel(BASE_CONFIG)
policy_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
policy_model.to(device)

reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
reference_model.to(device)
reference_model.eval()
```
**같은 파일을 두 번 읽어** 똑같은 모델 두 개를 만듭니다. 하나는 학습할 모델, 하나는 고정해 둘 기준입니다. 기준 모델은 `eval()` 로 평가 모드에 둡니다.

> ⚠️ **주석과 실제의 차이** : 주석은 "SFT 모델 로드"이지만, 불러오는 `gpt2-medium-355M.pth` 는 **미세 조정 전의 원본 GPT-2** 입니다. DPO 를 제대로 체험하려면 앞 노트북(`Chapter_7_Exercise_Follow_Instructions`)이 마지막에 저장하는 **SFT 완료 가중치**(`gpt2-medium355M-sft-standalone.pth`)를 불러와야 합니다. 그 파일이 준비되어 있다면 `model_path` 를 그 파일 경로로 바꿔 두 모델 모두에 사용하면 됩니다. (이 경우 `download_model` 줄은 필요 없습니다.)

> 💾 GPT-2 Medium 을 **두 개** 올리고 배치마다 네 번 실행하므로 GPU 메모리를 많이 씁니다. 메모리 부족(`CUDA out of memory`) 에러가 나면 `batch_size` 를 8 에서 4 나 2 로 줄여 보세요.

**④ 학습**
```python
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)

start_time = time.time()
num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model, reference_model=reference_model,
    train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, num_epochs=num_epochs,
    beta=0.1, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[2]), tokenizer=tokenizer
)
print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.")
```
- 옵티마이저에 **`policy_model.parameters()` 만** 넘깁니다 → 기준 모델은 학습되지 않습니다.
- `lr=5e-6` (0.000005) : 지금까지 중 **가장 작은 학습률**입니다. DPO 는 모델이 쉽게 망가질 수 있어 아주 조심스럽게 고칩니다. 에포크도 1번만 돕니다.
- `beta=0.1` : 셀 1의 DPO 손실에 쓰이는 민감도 값.

### 실행 결과 읽기
저장된 출력에는 `Using device: cuda` 한 줄만 남아 있습니다(학습 로그는 저장되어 있지 않음). 직접 실행하면 아래와 같은 **형식**의 줄이 5스텝마다 출력됩니다.

```
Ep 1 (Step 000000): Train loss 0.69x, Val loss 0.69x, Train Margin ...
```
이렇게 읽으면 됩니다.
- **손실** : 약 **0.693** 에서 시작해 **내려가면** 정상
- **Train Margin** : 0 근처에서 시작해 **커지면** 정상 (좋은 답과 나쁜 답을 점점 더 잘 구분)
- 끝에 학습된 모델이 쓴 샘플 글과 걸린 시간이 출력됩니다.

In [ ]:
# 2
# ==========================================
# 5. 메인 실행 코드
# ==========================================
# 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = tiktoken.get_encoding("gpt2")

# 데이터 로드
file_path = "data/instruction-data-with-preference.json"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
data = download_and_load_file(file_path, url)

# 데이터 분할
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

# DataLoader 생성
customized_collate_fn = partial(
    custom_collate_fn, device=device, mask_prompt_tokens=True, allowed_max_length=1024
)

batch_size = 8
train_loader = DataLoader(PreferenceDataset(train_data, tokenizer), batch_size=batch_size, 
                            collate_fn=customized_collate_fn, shuffle=True, drop_last=True)
val_loader = DataLoader(PreferenceDataset(val_data, tokenizer), batch_size=batch_size, 
                        collate_fn=customized_collate_fn, shuffle=False)

# 모델 설정 (SFT 모델 로드)
BASE_CONFIG = {
    "vocab_size": 50257, "context_length": 1024, "drop_rate": 0.0, "qkv_bias": True,
    "emb_dim": 1024, "n_layers": 24, "n_heads": 16 # gpt2-medium config
}

file_name = "gpt2-medium-355M.pth"
model_path = f"./models/gpt2/{file_name}"
download_model(file_name, model_path)

# Policy Model (학습 대상)
policy_model = GPTModel(BASE_CONFIG)
policy_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
policy_model.to(device)

# Reference Model (고정 기준점)
reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
reference_model.to(device)
reference_model.eval()

# 학습 시작
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)

start_time = time.time()
num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    beta=0.1,
    eval_freq=5,
    eval_iter=5,
    start_context=format_input(val_data[2]),
    tokenizer=tokenizer
)
print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.")

Using device: cuda


---
## 셀 3 — 손실 그래프

### 🎯 이 셀이 하는 일
셀 2에서 기록해 둔 손실을 그래프로 그립니다.

### 코드 뜯어보기
```python
from previous_chapters import plot_losses
num_epochs=1

epochs_tensor = torch.linspace(0, num_epochs, len(tracking["train_losses"]))
plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=tracking["train_losses"],
    val_losses=tracking["val_losses"],
    label="loss"
)
```
- `import` 는 꼭 맨 위가 아니라 **필요한 곳에서** 해도 됩니다.
- `tracking` 은 셀 2의 학습 함수가 돌려준 기록장 딕셔너리입니다. 그래서 **셀 2를 먼저 실행해야** 이 셀이 동작합니다(아니면 `NameError`).
- `torch.linspace(0, 1, n)` : 0~1 에포크 구간을 기록 개수만큼 고르게 나눈 x축 좌표
- 주석 처리된 `# from llms_from_scratch.ch05 import plot_losses` 는 같은 함수를 원저자의 패키지에서 가져오는 다른 방법을 메모해 둔 것입니다.

### 그래프 읽는 법
- 실선 = 학습 손실, 점선 = 검증 손실
- 두 선이 0.69 근처에서 시작해 **함께 내려가면** 잘 학습되고 있는 것입니다.

In [ ]:
# 3
from previous_chapters import plot_losses
# 대안:
# from llms_from_scratch.ch05 import plot_losses
num_epochs=1

epochs_tensor = torch.linspace(0, num_epochs, len(tracking["train_losses"]))
plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=tracking["train_losses"],
    val_losses=tracking["val_losses"],
    label="loss"
)

---
## 셀 4 — 기준 모델 vs 학습된 모델 답변 비교

### 🎯 이 셀이 하는 일
테스트 데이터 앞 3개에 대해 **기준 모델**과 **DPO 로 학습한 모델**이 각각 어떻게 답하는지 나란히 출력해 비교합니다.

### 코드 뜯어보기
```python
for entry in test_data[:3]:
    input_text = format_input(entry)

    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )
    ...  # policy_model 로 똑같이 한 번 더
```
- `test_data[:3]` : 앞에서 3개만
- 같은 프롬프트를 `reference_model` 과 `policy_model` 에 **각각** 넣어 답을 생성합니다. `model=` 만 다르고 나머지는 같습니다.
- `generated_text[len(input_text):]` : 생성 결과에서 프롬프트 부분을 잘라 내고 답변만 남깁니다.
- 괄호 안에서 `.replace(...)` `.strip()` 을 **줄을 바꿔 가며 이어 쓴** 것은 긴 연속 호출을 읽기 좋게 하려는 것입니다. 한 줄로 쓴 것과 같습니다.

```python
    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
```
프롬프트, 모범 답, 기준 모델의 답, 학습된 모델의 답을 차례로 출력합니다.

### 실행 결과 읽기 — 저장된 출력이 이상한 이유
```
### Instruction:
Rewrite the sentence using a simile.
### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Reference model response:
>> ### Output:  The car is very slow.  ### Instruction: ... (repeats)

Policy model response:
>> I would be happy to be corrected.  I would be happy to receive ... (repeats)
```
**두 모델 모두 지시를 따르지 못하고** 비슷한 글을 256토큰이 찰 때까지 반복합니다. 코드가 잘못 돌아간 것이 아니라, 셀 2 설명에 적은 대로 **지시 미세 조정(SFT)을 거치지 않은 원본 GPT-2** 를 출발점으로 썼기 때문입니다.
- 원본 GPT-2 는 `### Instruction:` 양식이 "답해야 할 지시"라는 것을 모르고, 답을 마치면 끝 토큰을 내야 한다는 것도 배운 적이 없습니다.
- DPO 는 "이미 답할 줄 아는 모델의 **말투를 다듬는**" 단계이지, 답하는 법 자체를 가르치는 단계가 아닙니다.

SFT 를 마친 가중치로 두 모델을 불러와 다시 실행하면, 기준 모델은 정답이지만 평범하거나 다소 무뚝뚝한 답을, 학습된 모델은 **조금 더 공손한 답**을 내는 식의 차이를 기대할 수 있습니다.

In [4]:
# 4
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Reference model response:
>> ### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:

The car is very fast.

### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:

The car is very fast.

### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:

The car is very fast.

### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:

The car is very fast.

### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:

---
## 셀 5 — (빈 셀)

번호(`# 5`)만 적혀 있고 코드는 없는 셀입니다. 직접 실험해 볼 자리로 쓰면 됩니다. 예를 들어 이런 것을 해 볼 수 있습니다.

```python
# 테스트 데이터 한 항목의 좋은 답 / 나쁜 답 확인
print(test_data[0]["chosen"])
print(test_data[0]["rejected"])

# 배치 하나의 모양 확인
batch = next(iter(train_loader))
print(batch["chosen"].shape, batch["chosen_mask"].shape)
```

In [ ]:
# 5


---
## 📌 DPO 정리

```
 data :  (prompt, chosen, rejected)

 for each batch :
     policy(chosen), policy(rejected)        trainable
     reference(chosen), reference(rejected)  fixed, no_grad
        |
        v  compute_logprobs   (average log-prob of the RESPONSE part only)
        v
     logits = (policy_chosen - policy_rejected) - (ref_chosen - ref_rejected)
     loss   = -logsigmoid(beta * logits)        starts at 0.693, should go down
```

**꼭 기억할 4가지**
1. DPO = **좋은 답 / 나쁜 답 쌍**을 보여 주고 좋은 쪽을 더 선호하게 만드는 학습.
2. 모델이 **2개** 필요하다 : 학습하는 policy, 고정된 reference. 기준 모델이 원래 능력을 붙잡아 준다.
3. 프롬프트와 패딩은 **마스크**로 가리고 **답변 부분만** 비교한다.
4. 출발점은 **SFT 를 마친 모델**이어야 의미 있는 결과가 나온다.